# 02 — Generation Agent: Llama-3.2-3B QLoRA SFT + JEPA

Inputs: SFT JSONL containing `question`, `schema` DDL, `answer` SQL and optional `think_steps`. Train completion-only cross entropy and optional latent JEPA alignment. Requires access to the gated base model (use Kaggle Secrets/HF authentication without printing tokens), or an attached local base model. Output: portable PEFT adapter, tokenizer and resumable optimizer/JEPA state.


In [ ]:
import os
import sys
from pathlib import Path

# Attach a repository snapshot as a Kaggle Dataset, or set KAGGLE_REPO_DIR.
_candidates = [Path(os.environ.get("KAGGLE_REPO_DIR", ".")), Path.cwd(), Path.cwd().parent]
_candidates += list(Path("/kaggle/input").glob("*/Optimize-Mars-Sl")) if Path("/kaggle/input").exists() else []
REPO_DIR = next((p.resolve() for p in _candidates if (p / "src").is_dir()), None)
if REPO_DIR is None:
    raise FileNotFoundError("Set KAGGLE_REPO_DIR to the attached repository directory containing src/")
sys.path.insert(0, str(REPO_DIR))
from src.training.kaggle import KagglePaths, install_dependencies, environment_report
install_dependencies(REPO_DIR)  # Only runs pip when INSTALL_DEPS=1.
paths = KagglePaths.from_env()
SMOKE = paths.smoke
print({"mode": "smoke (NOT a benchmark)" if SMOKE else "train", "data": str(paths.data_dir), "device": paths.device})


In [ ]:
PHASE = 'sft_jepa_3b'
import json
OUTPUT_DIR = paths.output(PHASE)
environment = environment_report(OUTPUT_DIR)
MAX_STEPS = int(os.environ.get("MAX_STEPS", "100"))
MAX_LENGTH = int(os.environ.get("MAX_LENGTH", "512"))
SEED = int(os.environ.get("SEED", "42"))
SAVE_STEPS = int(os.environ.get("SAVE_STEPS", "25"))
RESUME = os.environ.get("RESUME_FROM_CHECKPOINT") or None  # "latest" or attached checkpoint path
print({"output": OUTPUT_DIR, "resume_from_checkpoint": RESUME, "gpus": environment["gpus"]})


Set `SFT_DATA` and `GENERATOR_MODEL`. Use `LAMBDA_JEPA=0` for SFT-only; use the same seed, split and training budget for `LAMBDA_JEPA=0.1`. Use separate `KAGGLE_OUTPUT_DIR` values for ablations. Sequence overflows fail explicitly instead of silently removing the completion.

In [ ]:
from src.training.train_sft_generator import run_sft_training
SFT_DATA = paths.input("SFT_DATA", "mini_synsql.jsonl" if SMOKE else "sft_train.jsonl")
summary = run_sft_training(data_path=SFT_DATA, output_dir=OUTPUT_DIR, dry_run=SMOKE,
    model_name_or_path=os.environ.get("GENERATOR_MODEL", "meta-llama/Llama-3.2-3B-Instruct"),
    device=paths.device, max_steps=MAX_STEPS, max_length=MAX_LENGTH, save_steps=SAVE_STEPS,
    seed=SEED, gradient_accumulation_steps=int(os.environ.get("GRAD_ACCUM", "8")),
    lambda_jepa=float(os.environ.get("LAMBDA_JEPA", "0.1")),
    resume_from_checkpoint=RESUME)
print(json.dumps(summary, indent=2))


## Artifact handoff and validation

The last cell returns artifact paths. Download `/kaggle/working/optimize-mars` after a run or save a notebook version with outputs, then attach those outputs to the downstream notebook as a Kaggle Dataset. Runtime storage is ephemeral; a local checkpoint alone does not survive session deletion. Export before the session ends. Set `RESUME_FROM_CHECKPOINT` to an attached complete checkpoint, or `latest` for the current output directory. No checkpoints are automatically deleted.

Use `PIPELINE_MODE=smoke` for offline wiring checks; this mode does not train pretrained models or measure model quality. Default `train` requires real datasets, dependencies, GPU and the input artifacts listed above. Settings target **one 16 GB T4**, sequential rollouts and short sequences; T4 x2 does not provide a shared 32 GB memory pool. Reduce `MAX_LENGTH`, rollout length or accumulation if needed. No full GPU run or Kaggle benchmark is claimed by this notebook. See `KAGGLE.md` for data contracts, independent account handoffs and ablations.
